# Agent Orchestration

This notebook creates a *parent/orchestrator* agent that can delegate to two *subagents* defined in `src/agents.py`: 
- `orders_agent_setup()` (order Q&A)
- `product_agent_setup()` (Walmart product search/details)

In [1]:
import sys
sys.path.insert(0, "../src/")

import datetime
import dotenv

from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langchain.agents import create_agent

dotenv.load_dotenv("../.env")

import utils
import agents

## Subagents

In the **subagents** architecture, a central main [agent](/oss/python/langchain/agents) (often referred to as a **supervisor**) coordinates subagents by calling them as [tools](/oss/python/langchain/tools). 

The main agent decides which subagent to invoke, what input to provide, and how to combine results. 

Subagents are stateless—they don't remember past interactions, with all conversation memory maintained by the main agent. This provides [context](/oss/python/langchain/context-engineering) isolation: each subagent invocation works in a clean context window, preventing context bloat in the main conversation.


### Key characteristics

* Centralized control: All routing passes through the main agent
* No direct user interaction: Subagents return results to the main agent, not the user (though you can use [interrupts](/oss/python/langgraph/human-in-the-loop#interrupt) within a subagent to allow user interaction)
* Subagents via tools: Subagents are invoked via tools
* Parallel execution: The main agent can invoke multiple subagents in a single turn

<Note>
  **Supervisor vs. Router**: A supervisor agent (this pattern) is different from a [router](/oss/python/langchain/multi-agent/router). The supervisor is a full agent that maintains conversation context and dynamically decides which subagents to call across multiple turns. A router is typically a single classification step that dispatches to agents without maintaining ongoing conversation state.
</Note>

### When to use

Use the subagents pattern when you have multiple distinct domains (e.g., calendar, email, CRM, database), subagents don't need to converse directly with users, or you want centralized workflow control. For simpler cases with just a few [tools](/oss/python/langchain/tools), use a [single agent](/oss/python/langchain/agents).

<Tip>
  **Need user interaction within a subagent?** While subagents typically return results to the main agent rather than conversing directly with users, you can use [interrupts](/oss/python/langgraph/human-in-the-loop#interrupt) within a subagent to pause execution and gather user input. This is useful when a subagent needs clarification or approval before proceeding. The main agent remains the orchestrator, but the subagent can collect information from the user mid-task.
</Tip>

### Basic implementation

The core mechanism wraps a subagent as a tool that the main agent can call:

```python  theme={null}
from langchain.tools import tool
from langchain.agents import create_agent

# Create a subagent
subagent = create_agent(model="anthropic:claude-sonnet-4-20250514", tools=[...])

# Wrap it as a tool
@tool("research", description="Research a topic and return findings")
def call_research_agent(query: str):
    result = subagent.invoke({"messages": [{"role": "user", "content": query}]})
    return result["messages"][-1].content

# Main agent with subagent as a tool
main_agent = create_agent(model="anthropic:claude-sonnet-4-20250514", tools=[call_research_agent])
```

Reference:
https://docs.langchain.com/oss/python/langchain/multi-agent/subagents

In [4]:
utils.display_code(agents.orchestrator_agent_setup)

╭───────────────────────────────────────────── Function Code ─────────────────────────────────────────────╮
│    1 def orchestrator_agent_setup():                                                                    │
│    2     """Sets up the orchestrator agent that delegates to subagents."""                              │
│    3                                                                                                    │
│    4     # Create the two subagents                                                                     │
│    5     orders_subagent = orders_agent_setup()                                                         │
│    6     product_subagent = product_agent_setup()                                                       │
│    7                                                                                                    │
│    8     @tool                                                                                          │
│    9     def ask_orders_subagent(question: str, config: RunnableConfig) -> str:                         │
│   10         """Delegate order-related questions to the Orders subagent."""                             │
│   11         parent_thread_id = (config or {}).get("configurable", {}).get("thread_id", "default")      │
│   12         sub_config = {"configurable": {"thread_id": f"{parent_thread_id}:orders"}}                 │
│   13                                                                                                    │
│   14         resp = orders_subagent.invoke(                                                             │
│   15             {"messages": [HumanMessage(content=question)]},                                        │
│   16             config=sub_config,                                                                     │
│   17         )                                                                                          │
│   18         return resp["messages"][-1].content                                                        │
│   19                                                                                                    │
│   20     @tool                                                                                          │
│   21     def ask_product_subagent(question: str, config: RunnableConfig) -> str:                        │
│   22         """Delegate Walmart product search/details questions to the Product subagent."""           │
│   23         parent_thread_id = (config or {}).get("configurable", {}).get("thread_id", "default")      │
│   24         sub_config = {"configurable": {"thread_id": f"{parent_thread_id}:products"}}               │
│   25                                                                                                    │
│   26         resp = product_subagent.invoke(                                                            │
│   27             {"messages": [HumanMessage(content=question)]},                                        │
│   28             config[38;2;

In [5]:
orchestrator_agent = agents.orchestrator_agent_setup()

## Demo

In [ ]:
# Example: order question
resp1 = orchestrator_agent.invoke(
    {"messages": [HumanMessage(content="How many orders did I make in 2024?")]},
    config={"configurable": {"thread_id": "orchestrator_demo"}},
)
resp1["messages"][-1].content

In [ ]:
# Example: product question
resp2 = orchestrator_agent.invoke(
    {"messages": [HumanMessage(content="Find a good Bluetooth speaker under $50")]},
    config={"configurable": {"thread_id": "orchestrator_demo"}},
)
resp2["messages"][-1].content

In [ ]:
import rich

In [ ]:
rich.print(resp2["messages"][-1].content)

In [ ]:
resp3 = orchestrator_agent.invoke(
    {"messages": [HumanMessage(content="can u please compare and contrast across the bluetooth speaker options you found for me?")]},
    config={"configurable": {"thread_id": "orchestrator_demo"}},
)
rich.print(resp3["messages"][-1].content)